In [11]:
# -*- coding: utf-8 -*-
"""SLSL_DATASET_COMPLETE_FIXED.ipynb

COMPLETE FIXED VERSION:
- Proper extraction: Pose has visibility, Hands/Lip use padding (0.0)
- 468 features per frame (84 + 84 + 100 + 200)
- LIP landmarks (50 points for speech reading) - NOT full face
- Colored dots on black background
- Align to max frames
- Visualizations per signer
- Memory optimized
- FIXED: OpenCV cvtColor error with numpy object arrays
- ADDED: Frame count and video duration for each video
- UPDATED: Organized folder structure (Latest + Versions)
"""

# ============================================
# 1. INSTALLATIONS & IMPORTS
# ============================================

!pip install mediapipe==0.10.14 opencv-python pandas numpy tqdm matplotlib scipy -q

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import os
import gc
import glob
import random
import json
import shutil
from datetime import datetime
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

print("✅ Setup complete")

# ============================================
# 2. CONFIGURATION (UPDATED PATHS)
# ============================================

class Config:
    # Base Research Path
    BASE_PATH = "/content/drive/MyDrive/Research/SLSL_Medical_Dataset/"

    # Latest folder (current working data)
    LATEST_PATH = os.path.join(BASE_PATH, "Latest")
    PROCESSED_PATH = os.path.join(LATEST_PATH, "Original")      # Original aligned features
    VIZ_PATH = os.path.join(LATEST_PATH, "Visualization")       # Visualizations
    AUG_PATH = os.path.join(LATEST_PATH, "Augmented")           # Augmented features

    # Versions folder (archive)
    VERSIONS_PATH = os.path.join(BASE_PATH, "Versions")
    CURRENT_VERSION = datetime.now().strftime("%Y%m%d_%H%M%S")
    VERSION_FOLDER = os.path.join(VERSIONS_PATH, CURRENT_VERSION)
    VERSION_PROCESSED = os.path.join(VERSION_FOLDER, "Original")
    VERSION_VIZ = os.path.join(VERSION_FOLDER, "Visualization")
    VERSION_AUG = os.path.join(VERSION_FOLDER, "Augmented")

    # Raw videos path (input - unchanged)
    RAW_PATH = "/content/drive/MyDrive/Research/SLSL_Medical_Dataset_raw_videos_10/"

    # Feature counts (EXACTLY 468)
    LEFT_HAND_POINTS = 21
    RIGHT_HAND_POINTS = 21
    POSE_POINTS = 25
    LIP_POINTS = 50  # CHANGED: FACE to LIP

    LEFT_FEATURES = LEFT_HAND_POINTS * 4    # 84
    RIGHT_FEATURES = RIGHT_HAND_POINTS * 4  # 84
    POSE_FEATURES = POSE_POINTS * 4         # 100
    LIP_FEATURES = LIP_POINTS * 4           # 200
    TOTAL_FEATURES = LEFT_FEATURES + RIGHT_FEATURES + POSE_FEATURES + LIP_FEATURES  # 468

    # LIP landmarks indices (50 specific points around mouth for speech reading)
    LIP_INDICES = [
        61, 146, 91, 181, 84, 17, 314, 405, 321, 375,
        78, 95, 88, 178, 87, 14, 317, 402, 318, 324,
        308, 415, 310, 311, 312, 13, 82, 81, 80, 191,
        78, 76, 74, 73, 70, 63, 62, 96, 89, 179,
        86, 15, 316, 403, 319, 325, 307, 414, 309, 313
    ]

    # CRITICAL: Model complexity - MUST be 1 or 2 for hand detection
    MODEL_COMPLEXITY = 1  # 0=Lite (no hands), 1=Full, 2=Heavy

    # Detection thresholds
    MIN_DETECTION_CONFIDENCE = 0.5
    MIN_TRACKING_CONFIDENCE = 0.5

    # Colors for visualization (BGR format)
    COLORS = {
        'left_hand': (255, 0, 0),      # BLUE
        'right_hand': (0, 255, 0),     # GREEN
        'pose': (0, 0, 255),           # RED
        'lip': (0, 255, 255)           # YELLOW (for LIP)
    }

    # Smoothing
    SMOOTHING_WINDOW = 5

    # Memory
    CLEAR_MEMORY_EVERY = 3

cfg = Config()

# Create directory structure
def create_directory_structure():
    """Create organized directory structure"""

    # Create Latest subdirectories
    os.makedirs(cfg.PROCESSED_PATH, exist_ok=True)
    os.makedirs(cfg.VIZ_PATH, exist_ok=True)
    os.makedirs(cfg.AUG_PATH, exist_ok=True)

    # Create Version folder for this run
    os.makedirs(cfg.VERSION_PROCESSED, exist_ok=True)
    os.makedirs(cfg.VERSION_VIZ, exist_ok=True)
    os.makedirs(cfg.VERSION_AUG, exist_ok=True)

    print("="*60)
    print("DIRECTORY STRUCTURE CREATED")
    print("="*60)
    print(f"\n📁 LATEST (Current Data):")
    print(f"   ├── Original:      {cfg.PROCESSED_PATH}")
    print(f"   ├── Augmented:     {cfg.AUG_PATH}")
    print(f"   └── Visualization: {cfg.VIZ_PATH}")
    print(f"\n📁 VERSION (Archive):")
    print(f"   └── {cfg.CURRENT_VERSION}/")
    print(f"       ├── Original/")
    print(f"       ├── Augmented/")
    print(f"       └── Visualization/")
    print("="*60)

create_directory_structure()

# Clean Latest directory before processing
def clean_latest_directory():
    """Remove all existing files from Latest subdirectories"""

    print("\n" + "="*60)
    print("🧹 CLEANING LATEST DIRECTORY")
    print("="*60)

    for path in [cfg.PROCESSED_PATH, cfg.AUG_PATH, cfg.VIZ_PATH]:
        if os.path.exists(path):
            for item in os.listdir(path):
                item_path = os.path.join(path, item)
                if os.path.isfile(item_path):
                    os.remove(item_path)
                elif os.path.isdir(item_path):
                    shutil.rmtree(item_path)
            print(f"✅ Cleaned: {path}")

clean_latest_directory()

print("="*60)
print("CONFIGURATION")
print("="*60)
print(f"Total features per frame: {cfg.TOTAL_FEATURES}")
print(f"  - Left Hand: {cfg.LEFT_HAND_POINTS} points × 4 = {cfg.LEFT_FEATURES} (BLUE)")
print(f"  - Right Hand: {cfg.RIGHT_HAND_POINTS} points × 4 = {cfg.RIGHT_FEATURES} (GREEN)")
print(f"  - Pose: {cfg.POSE_POINTS} points × 4 = {cfg.POSE_FEATURES} (RED)")
print(f"  - LIP: {cfg.LIP_POINTS} points × 4 = {cfg.LIP_FEATURES} (YELLOW) - Speech Reading")
print(f"\nModel Complexity: {cfg.MODEL_COMPLEXITY} (1=Full model for hand detection)")
print(f"\n📁 Output Locations:")
print(f"   Original features: {cfg.PROCESSED_PATH}")
print(f"   Augmented features: {cfg.AUG_PATH}")
print(f"   Visualizations: {cfg.VIZ_PATH}")
print(f"   Version backup: {cfg.VERSION_FOLDER}")
print("="*60)

# ============================================
# 3. LANDMARK EXTRACTOR (LIP INSTEAD OF FACE)
# ============================================

class LandmarkExtractor:
    def __init__(self):
        self.mp_holistic = mp.solutions.holistic
        self.holistic = self.mp_holistic.Holistic(
            static_image_mode=False,
            model_complexity=cfg.MODEL_COMPLEXITY,
            smooth_landmarks=True,
            enable_segmentation=False,
            refine_face_landmarks=True,
            min_detection_confidence=cfg.MIN_DETECTION_CONFIDENCE,
            min_tracking_confidence=cfg.MIN_TRACKING_CONFIDENCE
        )
        self.mp_draw = mp.solutions.drawing_utils

    def extract_features(self, frame):
        """
        Extract 468 features:
        - Left Hand: 21 points × 4 (x,y,z,0) - NO visibility
        - Right Hand: 21 points × 4 (x,y,z,0) - NO visibility
        - Pose: 25 points × 4 (x,y,z,visibility) - HAS visibility
        - LIP: 50 points × 4 (x,y,z,0) - NO visibility (speech reading)
        """
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        results = self.holistic.process(rgb)

        # Initialize arrays (all zeros by default)
        left_hand_features = np.zeros(cfg.LEFT_FEATURES, dtype=np.float32)
        right_hand_features = np.zeros(cfg.RIGHT_FEATURES, dtype=np.float32)
        pose_features = np.zeros(cfg.POSE_FEATURES, dtype=np.float32)
        lip_features = np.zeros(cfg.LIP_FEATURES, dtype=np.float32)

        # Quality tracking
        quality = {
            'left': False, 'right': False, 'pose': False, 'lip': False,
            'left_pts': 0, 'right_pts': 0, 'pose_pts': 0, 'lip_pts': 0
        }

        # LEFT HAND - 21 points × 4 (x, y, z, 0.0)
        if results.left_hand_landmarks:
            cnt = 0
            for i, lm in enumerate(results.left_hand_landmarks.landmark[:21]):
                idx = i * 4
                left_hand_features[idx] = lm.x
                left_hand_features[idx + 1] = lm.y
                left_hand_features[idx + 2] = lm.z
                left_hand_features[idx + 3] = 0.0
                cnt += 1
            if cnt > 0:
                quality['left'] = True
                quality['left_pts'] = cnt

        # RIGHT HAND - 21 points × 4 (x, y, z, 0.0)
        if results.right_hand_landmarks:
            cnt = 0
            for i, lm in enumerate(results.right_hand_landmarks.landmark[:21]):
                idx = i * 4
                right_hand_features[idx] = lm.x
                right_hand_features[idx + 1] = lm.y
                right_hand_features[idx + 2] = lm.z
                right_hand_features[idx + 3] = 0.0
                cnt += 1
            if cnt > 0:
                quality['right'] = True
                quality['right_pts'] = cnt

        # POSE - 25 points × 4 (x, y, z, visibility)
        if results.pose_landmarks:
            cnt = 0
            for i, lm in enumerate(results.pose_landmarks.landmark[:25]):
                idx = i * 4
                pose_features[idx] = lm.x
                pose_features[idx + 1] = lm.y
                pose_features[idx + 2] = lm.z
                pose_features[idx + 3] = lm.visibility
                if lm.visibility > 0.3:
                    cnt += 1
            if cnt > 0:
                quality['pose'] = True
                quality['pose_pts'] = cnt

        # LIP - 50 points × 4 (x, y, z, 0.0) - ONLY LIP LANDMARKS
        if results.face_landmarks:
            cnt = 0
            for idx, i in enumerate(cfg.LIP_INDICES):
                if i < len(results.face_landmarks.landmark):
                    lm = results.face_landmarks.landmark[i]
                    feature_idx = idx * 4
                    lip_features[feature_idx] = lm.x
                    lip_features[feature_idx + 1] = lm.y
                    lip_features[feature_idx + 2] = lm.z
                    lip_features[feature_idx + 3] = 0.0
                    cnt += 1
            if cnt > 0:
                quality['lip'] = True
                quality['lip_pts'] = cnt

        # Combine all features: 84 + 84 + 100 + 200 = 468
        features = np.concatenate([
            left_hand_features,
            right_hand_features,
            pose_features,
            lip_features
        ])

        return features, results, quality

    def smooth_features(self, features_seq):
        """Apply moving average smoothing"""
        if len(features_seq) < cfg.SMOOTHING_WINDOW:
            return features_seq

        smoothed = np.zeros_like(features_seq)
        for i in range(features_seq.shape[1]):
            col = features_seq[:, i]
            if np.any(col != 0):
                smoothed[:, i] = np.convolve(col, np.ones(cfg.SMOOTHING_WINDOW)/cfg.SMOOTHING_WINDOW, mode='same')
            else:
                smoothed[:, i] = col
        return smoothed

    def normalize_pose(self, features):
        """Normalize pose coordinates relative to shoulders"""
        pose_start = cfg.LEFT_FEATURES + cfg.RIGHT_FEATURES
        pose_end = pose_start + cfg.POSE_FEATURES
        pose_flat = features[pose_start:pose_end]
        pose = pose_flat.reshape(cfg.POSE_POINTS, 4)

        left_shoulder = pose[11] if 11 < len(pose) else None
        right_shoulder = pose[12] if 12 < len(pose) else None

        if left_shoulder is not None and right_shoulder is not None and left_shoulder[3] > 0.3:
            shoulder_center_x = (left_shoulder[0] + right_shoulder[0]) / 2
            shoulder_center_y = (left_shoulder[1] + right_shoulder[1]) / 2
            shoulder_distance = abs(left_shoulder[0] - right_shoulder[0])

            if shoulder_distance > 0:
                for i in range(0, len(features), 4):
                    if features[i+3] != 0 or i >= pose_start:
                        features[i] = (features[i] - shoulder_center_x) / shoulder_distance
                        features[i+1] = (features[i+1] - shoulder_center_y) / shoulder_distance

        return features

    def draw_landmarks(self, frame, results):
        """Draw landmarks with colors on original frame"""
        annotated = frame.copy()
        h, w = frame.shape[:2]

        # LIP landmarks (YELLOW) - ONLY LIP, not full face
        if results.face_landmarks:
            for idx in cfg.LIP_INDICES:
                if idx < len(results.face_landmarks.landmark):
                    lm = results.face_landmarks.landmark[idx]
                    x, y = int(lm.x * w), int(lm.y * h)
                    cv2.circle(annotated, (x, y), 3, (0, 255, 255), -1)

        # Pose (RED)
        if results.pose_landmarks:
            self.mp_draw.draw_landmarks(
                annotated, results.pose_landmarks,
                self.mp_holistic.POSE_CONNECTIONS,
                landmark_drawing_spec=self.mp_draw.DrawingSpec(color=(0,0,255), thickness=2, circle_radius=3),
                connection_drawing_spec=self.mp_draw.DrawingSpec(color=(0,0,255), thickness=2))

        # Left Hand (BLUE)
        if results.left_hand_landmarks:
            for lm in results.left_hand_landmarks.landmark[:21]:
                x, y = int(lm.x * w), int(lm.y * h)
                cv2.circle(annotated, (x, y), 4, (255, 0, 0), -1)
            for conn in self.mp_holistic.HAND_CONNECTIONS:
                start, end = conn
                if start < 21 and end < 21:
                    s = results.left_hand_landmarks.landmark[start]
                    e = results.left_hand_landmarks.landmark[end]
                    x1, y1 = int(s.x * w), int(s.y * h)
                    x2, y2 = int(e.x * w), int(e.y * h)
                    cv2.line(annotated, (x1, y1), (x2, y2), (255, 0, 0), 2)

        # Right Hand (GREEN)
        if results.right_hand_landmarks:
            for lm in results.right_hand_landmarks.landmark[:21]:
                x, y = int(lm.x * w), int(lm.y * h)
                cv2.circle(annotated, (x, y), 4, (0, 255, 0), -1)
            for conn in self.mp_holistic.HAND_CONNECTIONS:
                start, end = conn
                if start < 21 and end < 21:
                    s = results.right_hand_landmarks.landmark[start]
                    e = results.right_hand_landmarks.landmark[end]
                    x1, y1 = int(s.x * w), int(s.y * h)
                    x2, y2 = int(e.x * w), int(e.y * h)
                    cv2.line(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)

        return annotated

    def create_colored_black_bg(self, frame_shape, results):
        """Black background with colored dots"""
        h, w = frame_shape[:2]
        black = np.zeros((h, w, 3), dtype=np.uint8)

        # Pose (RED - largest dots)
        if results.pose_landmarks:
            for lm in results.pose_landmarks.landmark[:25]:
                x, y = int(lm.x * w), int(lm.y * h)
                if 0 <= x < w and 0 <= y < h:
                    cv2.circle(black, (x, y), 8, cfg.COLORS['pose'], -1)

        # Left Hand (BLUE)
        if results.left_hand_landmarks:
            for lm in results.left_hand_landmarks.landmark[:21]:
                x, y = int(lm.x * w), int(lm.y * h)
                if 0 <= x < w and 0 <= y < h:
                    cv2.circle(black, (x, y), 6, cfg.COLORS['left_hand'], -1)

        # Right Hand (GREEN)
        if results.right_hand_landmarks:
            for lm in results.right_hand_landmarks.landmark[:21]:
                x, y = int(lm.x * w), int(lm.y * h)
                if 0 <= x < w and 0 <= y < h:
                    cv2.circle(black, (x, y), 6, cfg.COLORS['right_hand'], -1)

        # LIP landmarks (YELLOW - medium dots) - ONLY LIP
        if results.face_landmarks:
            for idx in cfg.LIP_INDICES:
                if idx < len(results.face_landmarks.landmark):
                    lm = results.face_landmarks.landmark[idx]
                    x, y = int(lm.x * w), int(lm.y * h)
                    if 0 <= x < w and 0 <= y < h:
                        cv2.circle(black, (x, y), 4, cfg.COLORS['lip'], -1)

        return black

    def close(self):
        self.holistic.close()
        gc.collect()

# ============================================
# 4. VIDEO PROCESSING (MEMORY OPTIMIZED) - UPDATED PATHS
# ============================================

def get_video_info(video_path):
    """Get video frame count and duration (supports multiple formats)"""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"    Warning: Could not open video file: {video_path}")
        return 0, 0, 0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    duration = total_frames / fps if fps > 0 else 0
    cap.release()
    return total_frames, fps, duration

def process_video(video_path, output_path, save_viz_samples=False):
    """Process one video - extract features, save to disk"""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"    ❌ Error: Cannot open video file: {video_path}")
        return {
            'frames': 0, 'fps': 0, 'duration': 0,
            'left_rate': 0, 'right_rate': 0, 'pose_rate': 0, 'lip_rate': 0,
            'left_pts': 0, 'right_pts': 0, 'pose_pts': 0, 'lip_pts': 0
        }

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    duration = total / fps if fps > 0 else 0

    print(f"    Frames: {total}, FPS: {fps:.2f}, Duration: {duration:.2f}s")

    extractor = LandmarkExtractor()
    features_list = []
    quality_list = []

    # For visualization samples
    sample_frames = []
    sample_annotated = []
    sample_blackbg = []

    with tqdm(total=total, desc="    Processing", leave=False) as pbar:
        frame_idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            features, results, quality = extractor.extract_features(frame)
            features_list.append(features)
            quality_list.append(quality)

            # Save sample frames for visualization (first 12)
            if save_viz_samples and len(sample_frames) < 12 and frame_idx % (max(1, total//12)) == 0:
                sample_frames.append(frame)
                sample_annotated.append(extractor.draw_landmarks(frame, results))
                sample_blackbg.append(extractor.create_colored_black_bg(frame.shape, results))

            frame_idx += 1
            pbar.update(1)

    cap.release()

    features_array = np.array(features_list, dtype=np.float32)

    # Apply smoothing
    features_array = extractor.smooth_features(features_array)

    # Apply normalization to all frames
    for i in range(len(features_array)):
        features_array[i] = extractor.normalize_pose(features_array[i])

    # Save features
    np.save(output_path, features_array)

    # Save visualization samples if requested
    if save_viz_samples and sample_frames:
        viz_dir = output_path.replace('.npy', '_viz')
        os.makedirs(viz_dir, exist_ok=True)
        for i, (orig, ann, bg) in enumerate(zip(sample_frames, sample_annotated, sample_blackbg)):
            cv2.imwrite(os.path.join(viz_dir, f'frame_{i:04d}_original.jpg'), orig)
            cv2.imwrite(os.path.join(viz_dir, f'frame_{i:04d}_annotated.jpg'), ann)
            cv2.imwrite(os.path.join(viz_dir, f'frame_{i:04d}_blackbg.jpg'), bg)

        sample_info = {
            'num_samples': len(sample_frames),
            'total_frames': total,
            'fps': fps,
            'duration': duration
        }
        with open(os.path.join(viz_dir, 'sample_info.json'), 'w') as f:
            json.dump(sample_info, f)

    extractor.close()

    # Calculate quality metrics
    left_rate = np.mean([q['left'] for q in quality_list]) * 100
    right_rate = np.mean([q['right'] for q in quality_list]) * 100
    pose_rate = np.mean([q['pose'] for q in quality_list]) * 100
    lip_rate = np.mean([q['lip'] for q in quality_list]) * 100

    left_pts = np.mean([q['left_pts'] for q in quality_list if q['left_pts'] > 0]) if any(q['left_pts'] > 0 for q in quality_list) else 0
    right_pts = np.mean([q['right_pts'] for q in quality_list if q['right_pts'] > 0]) if any(q['right_pts'] > 0 for q in quality_list) else 0
    pose_pts = np.mean([q['pose_pts'] for q in quality_list if q['pose_pts'] > 0]) if any(q['pose_pts'] > 0 for q in quality_list) else 0
    lip_pts = np.mean([q['lip_pts'] for q in quality_list if q['lip_pts'] > 0]) if any(q['lip_pts'] > 0 for q in quality_list) else 0

    print(f"\n    ✅ Results:")
    print(f"       Left Hand:  {left_rate:.1f}% ({left_pts:.1f}/21) | Right Hand: {right_rate:.1f}% ({right_pts:.1f}/21)")
    print(f"       Pose:       {pose_rate:.1f}% ({pose_pts:.1f}/25) | Lip: {lip_rate:.1f}% ({lip_pts:.1f}/50)")

    return {
        'frames': len(features_list),
        'fps': fps,
        'duration': duration,
        'left_rate': left_rate, 'right_rate': right_rate,
        'pose_rate': pose_rate, 'lip_rate': lip_rate,
        'left_pts': left_pts, 'right_pts': right_pts,
        'pose_pts': pose_pts, 'lip_pts': lip_pts
    }

def process_dataset(raw_path, processed_path, test_mode=False, max_videos=None):
    """Process all videos - memory optimized"""

    print("\n" + "="*60)
    print("STEP 1: PROCESSING VIDEOS")
    print("="*60)

    sentences = [d for d in os.listdir(raw_path) if os.path.isdir(os.path.join(raw_path, d))]
    all_metadata = []
    total = 0

    # Supported video formats
    video_extensions = ('.mp4', '.mov', '.avi', '.mkv', '.webm', '.flv', '.wmv', '.m4v')

    i = 1
    for sentence in sentences:
        sentence_path = os.path.join(raw_path, sentence)
        videos = [f for f in os.listdir(sentence_path)
                 if f.lower().endswith(video_extensions)]

        if test_mode:
            videos = videos[:2]

        print(f"\n📂 {i}. {sentence} ({len(videos)} videos)")
        i += 1

        out_dir = os.path.join(processed_path, sentence)
        os.makedirs(out_dir, exist_ok=True)

        for video in videos:
            if max_videos and total >= max_videos:
                break

            name_without_ext = video.rsplit('.', 1)[0]
            parts = name_without_ext.split('_')
            signer = parts[1] if len(parts) > 1 else "unknown"
            rep = parts[3] if len(parts) > 3 else "01"

            video_path = os.path.join(sentence_path, video)
            out_path = os.path.join(out_dir, f"signer_{signer}_rep_{rep}.npy")

            # Save visualization samples for first video of each signer
            save_viz = not any(m['signer'] == signer for m in all_metadata)

            print(f"\n  📹 {video}")
            result = process_video(video_path, out_path, save_viz_samples=save_viz)

            all_metadata.append({
                'sentence': sentence, 'signer': signer, 'rep': rep,
                'frames': result['frames'], 'fps': result['fps'], 'duration': result['duration'],
                'path': out_path, 'video_file': video,
                'left_rate': result['left_rate'], 'right_rate': result['right_rate'],
                'pose_rate': result['pose_rate'], 'lip_rate': result['lip_rate'],
                'left_pts': result['left_pts'], 'right_pts': result['right_pts'],
                'pose_pts': result['pose_pts'], 'lip_pts': result['lip_pts']
            })
            total += 1

            if total % cfg.CLEAR_MEMORY_EVERY == 0:
                gc.collect()

        if max_videos and total >= max_videos:
            break

    df = pd.DataFrame(all_metadata)
    df.to_csv(os.path.join(processed_path, "metadata.csv"), index=False)

    print(f"\n✅ Processed {total} videos")

    print("\n📊 VIDEO SUMMARY (with frame counts and durations):")
    for idx, row in df.iterrows():
        print(f"   {row['video_file']}: {row['frames']} frames, {row['duration']:.2f}s, Signer: {row['signer']}")

    return df

# ============================================
# 5. ALIGN TO MAX FRAMES (UNCHANGED)
# ============================================

def pad_to_max_frames(metadata_df):
    """Pad all videos to max frame count"""

    print("\n" + "="*60)
    print("STEP 2: ALIGNING TO MAX FRAMES")
    print("="*60)

    max_frames = metadata_df['frames'].max()
    print(f"📊 Maximum frames across ALL videos: {max_frames}")

    aligned_metadata = []

    for idx, row in tqdm(metadata_df.iterrows(), total=len(metadata_df), desc="Padding"):
        features = np.load(row['path'])
        current = features.shape[0]

        if current < max_frames:
            padding = np.zeros((max_frames - current, features.shape[1]), dtype=np.float32)
            padded = np.vstack([features, padding])
            aligned_path = row['path'].replace('.npy', '_aligned.npy')
            np.save(aligned_path, padded)
        else:
            aligned_path = row['path']

        aligned_metadata.append({
            **row.to_dict(),
            'aligned_path': aligned_path,
            'aligned_frames': max_frames,
            'original_frames': current,
            'padded': current < max_frames
        })

        if idx % 50 == 0:
            gc.collect()

    aligned_df = pd.DataFrame(aligned_metadata)
    aligned_df.to_csv(os.path.join(cfg.PROCESSED_PATH, "aligned_metadata.csv"), index=False)

    print(f"✅ Aligned {len(aligned_df)} videos to {max_frames} frames")
    return aligned_df, max_frames

# ============================================
# 6. VISUALIZE ONE VIDEO PER SIGNER (UPDATED FOR LIP)
# ============================================

def visualize_one_video(row, max_frames, viz_path):
    """Create all visualizations for one video"""

    print(f"\n{'='*60}")
    print(f"🎬 VISUALIZING: {row['sentence']} - Signer {row['signer']}")
    print(f"   Original frames: {row['original_frames']}, Duration: {row['duration']:.2f}s, FPS: {row['fps']:.2f}")
    print(f"{'='*60}")

    features = np.load(row['aligned_path'])

    out_dir = os.path.join(viz_path, row['sentence'], f"signer_{row['signer']}")
    os.makedirs(out_dir, exist_ok=True)

    # Save video info
    with open(os.path.join(out_dir, "video_info.txt"), 'w') as f:
        f.write(f"Video: {row['video_file']}\n")
        f.write(f"Sentence: {row['sentence']}\n")
        f.write(f"Signer: {row['signer']}\n")
        f.write(f"Original frames: {row['original_frames']}\n")
        f.write(f"Aligned frames: {max_frames}\n")
        f.write(f"Duration: {row['duration']:.2f} seconds\n")
        f.write(f"FPS: {row['fps']:.2f}\n")
        f.write(f"Padded: {row['padded']}\n")

    # Load saved visualization samples from JPG files
    viz_dir = row['path'].replace('.npy', '_viz')
    has_viz = os.path.exists(viz_dir)

    # ============================================
    # FIGURE 1: Frames with Landmarks + Colored Black BG
    # ============================================
    print("  📸 Creating frame visualizations...")

    if has_viz:
        annotated_files = sorted(glob.glob(os.path.join(viz_dir, '*_annotated.jpg')))
        blackbg_files = sorted(glob.glob(os.path.join(viz_dir, '*_blackbg.jpg')))

        if annotated_files and blackbg_files:
            n_frames = min(6, len(annotated_files))

            fig1, axes1 = plt.subplots(n_frames, 2, figsize=(14, 4*n_frames))
            fig1.suptitle(f"Landmark Extraction: {row['sentence']} - Signer {row['signer']}\n"
                          f"Original: {row['original_frames']} frames, {row['duration']:.2f}s | Aligned to: {max_frames} frames\n"
                          f"Left: With Landmarks | Right: Colored Dots on Black\n"
                          f"(BLUE=Left Hand, GREEN=Right Hand, RED=Pose, YELLOW=LIP (50 points - Speech))",
                          fontsize=12, fontweight='bold')

            for i in range(n_frames):
                annotated = cv2.imread(annotated_files[i])
                annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
                axes1[i, 0].imshow(annotated_rgb)
                axes1[i, 0].set_title(f"Frame {i} - With Landmarks", fontsize=9)
                axes1[i, 0].axis('off')

                blackbg = cv2.imread(blackbg_files[i])
                blackbg_rgb = cv2.cvtColor(blackbg, cv2.COLOR_BGR2RGB)
                axes1[i, 1].imshow(blackbg_rgb)
                axes1[i, 1].set_title(f"Frame {i} - Colored Dots (LIP in YELLOW)", fontsize=9)
                axes1[i, 1].axis('off')

            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, "01_frames_with_landmarks.png"), dpi=150, bbox_inches='tight')
            plt.show()
            print(f"  ✅ Saved: {out_dir}/01_frames_with_landmarks.png")

    # ============================================
    # FIGURE 2: Landmark Coordinates Over Time
    # ============================================
    print("  📈 Creating coordinate visualization...")

    fig2, axes2 = plt.subplots(2, 2, figsize=(14, 10))
    fig2.suptitle(f"Landmark Coordinates Over Time: {row['sentence']} - Signer {row['signer']}\n"
                  f"Original frames: {row['original_frames']} | Aligned to: {max_frames} frames | Duration: {row['duration']:.2f}s",
                  fontsize=12, fontweight='bold')

    frames = np.arange(max_frames)

    # Left wrist (feature 0)
    left_wrist = features[:, 0]
    axes2[0,0].plot(frames, left_wrist, 'b-', linewidth=1)
    axes2[0,0].axvline(x=row['original_frames'], color='r', linestyle='--', label='Original End')
    axes2[0,0].set_title("Left Hand Wrist X (BLUE)", fontsize=11)
    axes2[0,0].set_xlabel("Frame")
    axes2[0,0].set_ylabel("X Position")
    axes2[0,0].legend()
    axes2[0,0].grid(True, alpha=0.3)

    # Right wrist (feature 84)
    right_wrist = features[:, cfg.LEFT_FEATURES]
    axes2[0,1].plot(frames, right_wrist, 'g-', linewidth=1)
    axes2[0,1].axvline(x=row['original_frames'], color='r', linestyle='--', label='Original End')
    axes2[0,1].set_title("Right Hand Wrist X (GREEN)", fontsize=11)
    axes2[0,1].set_xlabel("Frame")
    axes2[0,1].set_ylabel("X Position")
    axes2[0,1].legend()
    axes2[0,1].grid(True, alpha=0.3)

    # Lip center (first lip feature)
    lip_start = cfg.LEFT_FEATURES + cfg.RIGHT_FEATURES + cfg.POSE_FEATURES
    lip_center = features[:, lip_start]
    axes2[1,0].plot(frames, lip_center, 'y-', linewidth=1, color='gold')
    axes2[1,0].axvline(x=row['original_frames'], color='r', linestyle='--', label='Original End')
    axes2[1,0].set_title("Lip Center X (YELLOW) - Speech Reading", fontsize=11)
    axes2[1,0].set_xlabel("Frame")
    axes2[1,0].set_ylabel("X Position")
    axes2[1,0].legend()
    axes2[1,0].grid(True, alpha=0.3)

    # Detection rates bar chart
    rates = [row['left_rate'], row['right_rate'], row['pose_rate'], row['lip_rate']]
    colors = ['blue', 'green', 'red', 'gold']
    labels = ['Left Hand', 'Right Hand', 'Pose', 'Lip']

    bars = axes2[1,1].bar(labels, rates, color=colors, alpha=0.7)
    axes2[1,1].set_title("Detection Rates", fontsize=11)
    axes2[1,1].set_ylabel("Rate (%)")
    axes2[1,1].set_ylim([0, 100])
    axes2[1,1].grid(True, alpha=0.3, axis='y')

    for i, v in enumerate(rates):
        axes2[1,1].text(i, v + 2, f"{v:.1f}%", ha='center')

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "02_coordinates_over_time.png"), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  ✅ Saved: {out_dir}/02_coordinates_over_time.png")

    # ============================================
    # FIGURE 3: Augmentations
    # ============================================
    print("  🔄 Creating augmentation visualization...")

    def add_noise(x): return x + np.random.normal(0, 0.01, x.shape)
    def scale(x):
        s = np.random.uniform(0.95, 1.05)
        scaled = x.copy()
        for i in range(0, x.shape[1], 4):
            scaled[:, i:i+3] *= s
        return scaled
    def rotate(x):
        angle = np.radians(np.random.uniform(-5, 5))
        rot = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
        rotated = x.copy()
        for i in range(0, x.shape[1], 4):
            xy = x[:, i:i+2]
            rotated[:, i:i+2] = np.dot(xy, rot.T)
        return rotated

    noise = add_noise(features)
    scaled = scale(features)
    rotated = rotate(features)
    combined = rotate(scale(add_noise(features)))

    fig3, axes3 = plt.subplots(2, 3, figsize=(18, 8))
    fig3.suptitle(f"Data Augmentation Effects: {row['sentence']} - Signer {row['signer']}",
                  fontsize=12, fontweight='bold')

    augs = [('Original', features), ('Gaussian Noise', noise), ('Scale', scaled),
            ('Rotation', rotated), ('Combined', combined)]

    for i, (title, aug_features) in enumerate(augs):
        row_i, col_i = i // 3, i % 3
        axes3[row_i, col_i].plot(frames[:row['original_frames']], aug_features[:row['original_frames'], 0],
                                 'b-', label='Left Wrist', linewidth=1)
        axes3[row_i, col_i].plot(frames[:row['original_frames']], aug_features[:row['original_frames'], cfg.LEFT_FEATURES],
                                 'g-', label='Right Wrist', linewidth=1)
        axes3[row_i, col_i].set_title(title)
        axes3[row_i, col_i].set_xlabel("Frame")
        axes3[row_i, col_i].set_ylabel("X Position")
        axes3[row_i, col_i].legend()
        axes3[row_i, col_i].grid(True, alpha=0.3)

    axes3[1,2].axis('off')
    axes3[1,2].text(0.1, 0.5,
                    "AUGMENTATION PARAMETERS:\n\n"
                    "• Gaussian Noise Std: 0.01\n"
                    "• Scale Range: 0.95 - 1.05\n"
                    "• Rotation: ±5 degrees\n\n"
                    "COLOR LEGEND:\n"
                    "• BLUE  = Left Hand\n"
                    "• GREEN = Right Hand\n"
                    "• RED   = Pose\n"
                    "• YELLOW = Lip (50 points - Speech)\n\n"
                    "PURPOSE:\n"
                    "Increase dataset size for\n"
                    "better model generalization",
                    fontsize=9, fontfamily='monospace',
                    bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "03_augmentations.png"), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  ✅ Saved: {out_dir}/03_augmentations.png")

    # ============================================
    # FIGURE 4: Quality Summary
    # ============================================
    print("  📊 Creating quality summary...")

    fig4, axes4 = plt.subplots(1, 2, figsize=(14, 5))
    fig4.suptitle(f"Quality Summary: {row['sentence']} - Signer {row['signer']}", fontsize=12, fontweight='bold')

    # Points detected
    points = [row['left_pts'], row['right_pts'], row['pose_pts'], row['lip_pts']]
    max_pts = [21, 21, 25, 50]

    x = np.arange(len(labels))
    bars1 = axes4[0].bar(x, points, width=0.6, color=colors, alpha=0.7, label='Detected')
    axes4[0].set_xticks(x)
    axes4[0].set_xticklabels(labels)
    axes4[0].set_title("Average Points Detected per Frame")
    axes4[0].set_ylabel("Number of Points")
    axes4[0].grid(True, alpha=0.3, axis='y')

    for i, (p, m) in enumerate(zip(points, max_pts)):
        axes4[0].text(i, p + 1, f"{p:.1f}/{m}", ha='center', fontsize=9)

    # Text summary
    axes4[1].axis('off')
    summary_text = f"""
    QUALITY ENHANCEMENTS APPLIED
    {'='*40}

    Model Complexity: {cfg.MODEL_COMPLEXITY} (Full model for hand detection)
    Smoothing: Moving average (window={cfg.SMOOTHING_WINDOW})
    Normalization: Centered to shoulders

    CRITICAL FIX:
    • Pose: Uses visibility (4 values)
    • Hands/Lip: No visibility (padded with 0.0)
    • LIP: 50 specific points for speech reading

    DETECTION RESULTS:
    {'='*40}
    Left Hand:  {row['left_rate']:.1f}% ({row['left_pts']:.1f}/21 points)
    Right Hand: {row['right_rate']:.1f}% ({row['right_pts']:.1f}/21 points)
    Pose:       {row['pose_rate']:.1f}% ({row['pose_pts']:.1f}/25 points)
    Lip:        {row['lip_rate']:.1f}% ({row['lip_pts']:.1f}/50 points)

    VIDEO INFORMATION:
    {'='*40}
    Video: {row['video_file']}
    Original frames: {row['original_frames']}
    Aligned frames: {max_frames}
    Duration: {row['duration']:.2f} seconds
    FPS: {row['fps']:.2f}
    Padding required: {'Yes' if row['padded'] else 'No'}

    COLOR CODING:
    • BLUE  = Left Hand
    • GREEN = Right Hand
    • RED   = Pose (Upper Body)
    • YELLOW = Lip (50 points - Speech Reading)
    """
    axes4[1].text(0.05, 0.5, summary_text, fontsize=9, fontfamily='monospace',
                  verticalalignment='center', transform=axes4[1].transAxes,
                  bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "04_quality_summary.png"), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  ✅ Saved: {out_dir}/04_quality_summary.png")

    print(f"\n  ✅ All visualizations complete for Signer {row['signer']}")

def visualize_all_signers(aligned_df, max_frames, viz_path, target_sentence=None, max_signers=5):
    """Visualize ONE specific sentence for ALL signers (max 5 signers)"""

    print("\n" + "="*60)
    print("STEP 3: VISUALIZING ONE SENTENCE FOR ALL SIGNERS")
    print("="*60)

    # If no target sentence specified, use the first sentence in the dataset
    if target_sentence is None:
        target_sentence = aligned_df['sentence'].iloc[0]
        print(f"📝 No sentence specified, using first sentence: {target_sentence}")
    else:
        print(f"📝 Target sentence: {target_sentence}")

    # Filter dataframe for the specific sentence
    sentence_df = aligned_df[aligned_df['sentence'] == target_sentence]

    if len(sentence_df) == 0:
        print(f"❌ Sentence '{target_sentence}' not found in dataset!")
        print(f"📋 Available sentences: {aligned_df['sentence'].unique()}")
        return

    # Get all unique signers for this sentence
    signer_videos = sentence_df.groupby('signer').first().reset_index()
    total_signers = len(signer_videos)

    # Limit to max_signers
    signer_videos = signer_videos.head(max_signers)
    num_to_viz = len(signer_videos)

    print(f"\n📊 Found {total_signers} unique signers for sentence: {target_sentence}")
    print(f"🎨 Visualizing {num_to_viz} signers (max {max_signers})...")

    for idx, row in signer_videos.iterrows():
        print(f"\n{'='*60}")
        print(f"📹 Processing signer {idx+1}/{num_to_viz}: {row['signer']}")
        print(f"{'='*60}")
        visualize_one_video(row, max_frames, viz_path)

    print(f"\n✅ Visualized {num_to_viz} signers for sentence: {target_sentence}")

    # Print summary of visualized signers
    print("\n📋 Visualized signers:")
    for idx, row in signer_videos.iterrows():
        print(f"   • Signer {row['signer']} - Frames: {row['original_frames']}, Duration: {row['duration']:.2f}s")

# ============================================
# 7. CREATE AUGMENTED DATASET (UNCHANGED)
# ============================================

def create_augmented_dataset(aligned_df):
    """Create augmented versions on disk"""

    print("\n" + "="*60)
    print("STEP 4: CREATING AUGMENTED DATASET")
    print("="*60)

    def add_noise(x): return x + np.random.normal(0, 0.01, x.shape)
    def scale(x):
        s = np.random.uniform(0.95, 1.05)
        scaled = x.copy()
        for i in range(0, x.shape[1], 4):
            scaled[:, i:i+3] *= s
        return scaled
    def rotate(x):
        angle = np.radians(np.random.uniform(-5, 5))
        rot = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
        rotated = x.copy()
        for i in range(0, x.shape[1], 4):
            xy = x[:, i:i+2]
            rotated[:, i:i+2] = np.dot(xy, rot.T)
        return rotated

    augmented_metadata = []

    for idx, row in tqdm(aligned_df.iterrows(), total=len(aligned_df), desc="Augmenting"):
        features = np.load(row['aligned_path'])

        out_dir = os.path.join(cfg.AUG_PATH, row['sentence'])
        os.makedirs(out_dir, exist_ok=True)

        # Original
        orig_path = os.path.join(out_dir, f"signer_{row['signer']}_rep_{row['rep']}_original.npy")
        np.save(orig_path, features)
        augmented_metadata.append({**row.to_dict(), 'aug_path': orig_path, 'type': 'original'})

        # Noise
        noise_path = os.path.join(out_dir, f"signer_{row['signer']}_rep_{row['rep']}_noise.npy")
        np.save(noise_path, add_noise(features))
        augmented_metadata.append({**row.to_dict(), 'aug_path': noise_path, 'type': 'noise'})

        # Scale
        scale_path = os.path.join(out_dir, f"signer_{row['signer']}_rep_{row['rep']}_scale.npy")
        np.save(scale_path, scale(features))
        augmented_metadata.append({**row.to_dict(), 'aug_path': scale_path, 'type': 'scale'})

        # Rotation
        rotate_path = os.path.join(out_dir, f"signer_{row['signer']}_rep_{row['rep']}_rotation.npy")
        np.save(rotate_path, rotate(features))
        augmented_metadata.append({**row.to_dict(), 'aug_path': rotate_path, 'type': 'rotation'})

        # Combined
        combined = rotate(scale(add_noise(features)))
        combined_path = os.path.join(out_dir, f"signer_{row['signer']}_rep_{row['rep']}_combined.npy")
        np.save(combined_path, combined)
        augmented_metadata.append({**row.to_dict(), 'aug_path': combined_path, 'type': 'combined'})

        if idx % 20 == 0:
            gc.collect()

    aug_df = pd.DataFrame(augmented_metadata)
    aug_df.to_csv(os.path.join(cfg.AUG_PATH, "augmented_metadata.csv"), index=False)

    print(f"\n✅ Created {len(aug_df)} augmented samples")
    print(f"   Original videos: {len(aligned_df)}")
    print(f"   Augmented videos: {len(aug_df) - len(aligned_df)}")

    return aug_df

# ============================================
# 8. CREATE VERSION BACKUP (NEW)
# ============================================

def create_version_backup():
    """Copy Latest data to Version folder"""

    print("\n" + "="*60)
    print("STEP 5: CREATING VERSION BACKUP")
    print("="*60)

    # Copy Original to Version
    if os.path.exists(cfg.PROCESSED_PATH):
        shutil.copytree(cfg.PROCESSED_PATH, cfg.VERSION_PROCESSED, dirs_exist_ok=True)
        print(f"✅ Backed up Original to: {cfg.VERSION_PROCESSED}")

    # Copy Augmented to Version
    if os.path.exists(cfg.AUG_PATH):
        shutil.copytree(cfg.AUG_PATH, cfg.VERSION_AUG, dirs_exist_ok=True)
        print(f"✅ Backed up Augmented to: {cfg.VERSION_AUG}")

    # Copy Visualization to Version
    if os.path.exists(cfg.VIZ_PATH):
        shutil.copytree(cfg.VIZ_PATH, cfg.VERSION_VIZ, dirs_exist_ok=True)
        print(f"✅ Backed up Visualization to: {cfg.VERSION_VIZ}")

    # Save version info
    version_info = {
        'version_id': cfg.CURRENT_VERSION,
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'description': 'Full dataset with original, augmented, and visualization data'
    }

    with open(os.path.join(cfg.VERSION_FOLDER, "version_info.json"), 'w') as f:
        json.dump(version_info, f, indent=2)

    # Save current version marker in Latest folder
    with open(os.path.join(cfg.LATEST_PATH, "current_version.txt"), 'w') as f:
        f.write(cfg.CURRENT_VERSION)

    print(f"\n✅ Version {cfg.CURRENT_VERSION} created successfully!")

# ============================================
# 9. RUN FULL PIPELINE
# ============================================

print("\n" + "="*60)
print("🚀 STARTING COMPLETE PIPELINE")
print("="*60)

# STEP 1: Process videos
metadata_df = process_dataset(cfg.RAW_PATH, cfg.PROCESSED_PATH, test_mode=False, max_videos=None)

# STEP 2: Align to max frames
aligned_df, max_frames = pad_to_max_frames(metadata_df)

# STEP 3: Visualize one video per signer
visualize_all_signers(aligned_df, max_frames, cfg.VIZ_PATH)

# STEP 4: Create augmented dataset
augmented_df = create_augmented_dataset(aligned_df)

# STEP 5: Create version backup
create_version_backup()

# ============================================
# 10. FINAL SUMMARY
# ============================================

print("\n" + "="*60)
print("✅ PIPELINE COMPLETE!")
print("="*60)
print(f"\n📊 FINAL SUMMARY:")
print(f"   Videos processed: {len(metadata_df)}")
print(f"   Max frames: {max_frames}")
print(f"   All videos aligned to: {max_frames} frames")
print(f"   Augmented samples: {len(augmented_df)}")

print(f"\n📁 DIRECTORY STRUCTURE:")
print(f"\n   📂 {cfg.BASE_PATH}")
print(f"   │")
print(f"   ├── 📁 Latest/                      (Current working data)")
print(f"   │   ├── 📁 Original/                {len(metadata_df)} videos")
print(f"   │   ├── 📁 Augmented/               {len(augmented_df)} samples")
print(f"   │   └── 📁 Visualization/           Visualizations")
print(f"   │")
print(f"   └── 📁 Versions/                    (Archived versions)")
print(f"        └── 📁 {cfg.CURRENT_VERSION}/  (This run)")
print(f"            ├── 📁 Original/")
print(f"            ├── 📁 Augmented/")
print(f"            └── 📁 Visualization/")

print(f"\n📊 DETECTION RATES:")
print(f"   Left Hand:  {metadata_df['left_rate'].mean():.1f}% ({metadata_df['left_pts'].mean():.1f}/21)")
print(f"   Right Hand: {metadata_df['right_rate'].mean():.1f}% ({metadata_df['right_pts'].mean():.1f}/21)")
print(f"   Pose:       {metadata_df['pose_rate'].mean():.1f}% ({metadata_df['pose_pts'].mean():.1f}/25)")
print(f"   Lip:        {metadata_df['lip_rate'].mean():.1f}% ({metadata_df['lip_pts'].mean():.1f}/50)")

print(f"\n📁 OUTPUT LOCATIONS:")
print(f"   Original features: {cfg.PROCESSED_PATH}")
print(f"   Visualizations: {cfg.VIZ_PATH}")
print(f"   Augmented: {cfg.AUG_PATH}")
print(f"   Version backup: {cfg.VERSION_FOLDER}")

print("\n🎨 COLOR CODING:")
print("   • BLUE  = Left Hand")
print("   • GREEN = Right Hand")
print("   • RED   = Pose (Upper Body)")
print("   • YELLOW = Lip (50 points - Speech Reading)")

print("\n✨ CRITICAL FIX APPLIED:")
print("   • Pose: Uses visibility (4 values)")
print("   • Hands/Lip: No visibility (padded with 0.0)")
print("   • LIP: 50 specific points for speech reading (not full face)")
print("   • OpenCV error fixed: Using JPG files instead of numpy object arrays")
print("   • Added: Frame count and video duration for each video")
print("   • Organized: Latest + Version folder structure")

print(f"\n📌 Current Version: {cfg.CURRENT_VERSION}")
print("="*60)

Output hidden; open in https://colab.research.google.com to view.